<a href="https://colab.research.google.com/github/kartikirasane27/31_Kartiki-Rasane/blob/main/Assignment7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
!pip install torch torchvision -q

# Import libraries
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import (
    alexnet, AlexNet_Weights,
    vgg16, VGG16_Weights,
    resnet50, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)
import pandas as pd
import time

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# Load CIFAR-10
train_data = torchvision.datasets.CIFAR10(
    "./data",
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    "./data",
    train=False,
    download=True,
    transform=transform
)

# Use small datasets for faster training
train_data = torch.utils.data.Subset(train_data, range(1000))
test_data = torch.utils.data.Subset(test_data, range(200))

# Data loaders
train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)

# Number of classes
num_classes = 10


# Create AlexNet
def get_alexnet():

    model = alexnet(weights=AlexNet_Weights.DEFAULT)

    for param in model.features.parameters():
        param.requires_grad = False

    model.classifier[6] = nn.Linear(
        model.classifier[6].in_features,
        num_classes
    )

    return model


# Create VGG16
def get_vgg16():

    model = vgg16(weights=VGG16_Weights.DEFAULT)

    for param in model.features.parameters():
        param.requires_grad = False

    model.classifier[6] = nn.Linear(
        model.classifier[6].in_features,
        num_classes
    )

    return model


# Create ResNet50
def get_resnet50():

    model = resnet50(weights=ResNet50_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    model.fc = nn.Linear(
        model.fc.in_features,
        num_classes
    )

    return model


# Create EfficientNetB0
def get_efficientnet():

    model = efficientnet_b0(
        weights=EfficientNet_B0_Weights.DEFAULT
    )

    for param in model.features.parameters():
        param.requires_grad = False

    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        num_classes
    )

    return model


# Train and evaluate model
def train_model(model):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )

    start = time.time()

    # One epoch
    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    # Evaluate
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    time_taken = time.time() - start

    return accuracy, time_taken


# Models to compare
models = {
    "AlexNet": get_alexnet,
    "VGG16": get_vgg16,
    "ResNet50": get_resnet50,
    "EfficientNetB0": get_efficientnet
}

# Store results
results = []

# Train each model
for name, create_model in models.items():

    print("\nTraining:", name)

    model = create_model()

    accuracy, time_taken = train_model(model)

    results.append([
        name,
        round(accuracy, 2),
        round(time_taken, 2)
    ])

    print("Accuracy:", round(accuracy, 2), "%")
    print("Time:", round(time_taken, 2), "seconds")


# Display comparison
results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy (%)",
        "Time (seconds)"
    ]
)

print("\nPerformance Comparison")
display(results)

# Find best model
best = results.loc[
    results["Accuracy (%)"].idxmax()
]

print(
    "\nBest Model:",
    best["Model"]
)

print(
    "Best Accuracy:",
    best["Accuracy (%)"],
    "%"
)

Device: cpu

Training: AlexNet
Accuracy: 57.5 %
Time: 79.88 seconds

Training: VGG16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:12<00:00, 43.9MB/s]
